# Frequent Actor Groups in the IMDb Top 1000 Movies

**Algorithms for Massive Datasets** — Master in Data Science for Economics, Università degli Studi di Milano, A.Y. 2025/26

**Project:** market-basket analysis. Each movie is a *basket* and its four billed stars are the *items*. The goal is to find groups of actors that frequently appear together, using techniques from the course (RU chapter 6), and to study how these techniques scale with the size of the data.

### Methods
| Method | Reference | Role in this notebook |
|---|---|---|
| Exhaustive counting | — | Ground truth: counts every subset of every basket |
| A-Priori (basic) | RU §6.2 | Textbook algorithm, one pass per itemset size |
| A-Priori (optimized) | RU §6.2 | Same algorithm, cheaper candidate generation and counting |
| PCY | RU §6.3.1 | Hash-based filter that reduces the pairs counted in the second pass |

Every algorithm is checked against exhaustive counting on every run.

### Structure
0. Setup
1. Dataset and the parts considered
2. Data organisation
3. Pre-processing
4. Algorithms and implementations
5. Experimental setup
6. Results on the original data
7. Scalability
8. Discussion and conclusions

In [1]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/amd-market-basket-imdb").exists():
        !git clone --quiet --depth 1 --branch main https://github.com/TheRealMamoot/amd-market-basket-imdb.git /content/amd-market-basket-imdb
    %cd /content/amd-market-basket-imdb
    !pip install --quiet -r requirements.txt
    sys.path.insert(0, str(Path.cwd()))

### Global settings

All parameters that control the experiments are defined once in the cell below and passed explicitly to the `src` functions in later sections, so every result can be reproduced by changing a single cell.

In [2]:
# Support thresholds, as numbers of movies, for the experiments on the original data
SUPPORTS = [1, 2, 3, 5, 10]

# Timed runs per configuration; runtimes are summarised by their median
REPEATS = 5

# Numbers of hash buckets tried for PCY
BUCKETS = [1_000, 10_000, 100_000, 1_000_000]

# Dataset sizes for the scalability experiments, as numbers of copies of the original data
COPY_COUNTS = [1, 10, 50, 100]

# Largest number of relabelled copies used for basic A-Priori, whose cost grows quadratically there
BASIC_MAX_COPIES = 10

## 1. Dataset and the parts considered

The data is the [IMDB Movies Dataset](https://www.kaggle.com/datasets/harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows) from Kaggle: the 1000 top-rated movies and TV shows on IMDb, one row per title. It is downloaded with the Kaggle API, so an API token is needed:

- **on Colab**, add a secret named `KAGGLE_API_TOKEN` in the *Secrets* panel (key icon on the left) and allow this notebook to access it;
- **locally**, put `KAGGLE_API_TOKEN=...` in the project's `.env` file (see `.env.example`), which is excluded from version control.

The token is never written in the notebook. If the dataset has already been downloaded, it is reused and no token is needed.

In [3]:
import os

if IN_COLAB:
    from google.colab import userdata

    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [4]:
from src.data import download_dataset

dataset_path = download_dataset()
print(f"Dataset: {dataset_path}")

Dataset: /Users/mahmoudian/Projects/amd-market-basket-imdb/data/imdb_top_1000.csv


### A first look at the raw data

In [5]:
from src.preprocessing import load_movies

movies = load_movies(dataset_path)
print(f"{movies.shape[0]} titles, {movies.shape[1]} columns")
movies.head()

1000 titles, 16 columns


,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


### Parts of the dataset considered

Following the project specification, each movie is a basket whose items are its four billed stars, so only the columns **`Star1`–`Star4`** are used to build the baskets. `Series_Title` is kept only to interpret the results. The remaining columns (genre, director, ratings, votes, gross, …) are not used.

The whole dataset is used; no subsampling is needed at this size.

In [6]:
movies[["Series_Title", "Star1", "Star2", "Star3", "Star4"]].head()

,Series_Title,Star1,Star2,Star3,Star4
0,The Shawshank Redemption,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler
1,The Godfather,Marlon Brando,Al Pacino,James Caan,Diane Keaton
2,The Dark Knight,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine
3,The Godfather: Part II,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton
4,12 Angry Men,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler
